In [ ]:
# !pip install pandas
# !pip install numpy
# !pip install matplotlib
# !pip install geopandas
# !pip install geoalchemy2
# !pip install sqlalchemy

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, Polygon, MultiPolygon
from geoalchemy2 import Geometry, WKTElement
import matplotlib.pyplot as plt
import os
import glob

In [ ]:
# !pip install psycopg2
# !pip install psycopg2-binary --user
# !pip install psycopg[binary]

from sqlalchemy import create_engine
import psycopg2
import psycopg2.extras
import json

credentials = "/Users/apple/Desktop/Data2901/Credentials.json"

def pgconnect(credential_filepath, db_schema="public"):
    with open(credential_filepath) as f:
        db_conn_dict = json.load(f)
        host       = db_conn_dict['host']
        db_user    = db_conn_dict['user']
        db_pw      = db_conn_dict['password']
        default_db = db_conn_dict['user']
        port       = db_conn_dict['port']
        try:
            db = create_engine(f'postgresql+psycopg2://{db_user}:{db_pw}@{host}:{port}/{default_db}', echo=False)
            conn = db.connect()
            print('Connected successfully.')
        except Exception as e:
            print("Unable to connect to the database.")
            print(e)
            db, conn = None, None
        return db,conn

def query(conn, sqlcmd, args=None, df=True):
    result = pd.DataFrame() if df else None
    try:
        if df:
            result = pd.read_sql_query(sqlcmd, conn, params=args)
        else:
            result = conn.execute(text(sqlcmd), args).fetchall()
            result = result[0] if len(result) == 1 else result
    except Exception as e:
        print("Error encountered: ", e, sep='\n')
    return result

In [ ]:
db, conn = pgconnect(credentials)

In [ ]:
query(conn, "select PostGIS_Version()")

# 1. import all the data

In [ ]:
# get all the files in csv
path = os.getcwd() 
csv_files = glob.glob(os.path.join(path, "*/*.csv")) 

print(csv_files)

# create empty list
dataframes_list = []
 
# append datasets into the list
for i in csv_files:
    temp_df = pd.read_csv(i)
    dataframes_list.append(temp_df)

df_income = dataframes_list[0]
df_pop = dataframes_list[1]
df_pol = dataframes_list[2]
df_business = dataframes_list[3]
print("Income: ",df_income.shape,'; ',df_income.columns)
print('Population: ',df_pop.shape,'; ',df_pop.columns)
print('Polling: ',df_pol.shape,'; ',df_pol.columns)
print('Business: ',df_business.shape,'; ',df_business.columns)

In [ ]:
df_income.head()

In [ ]:
df_pop.head()

In [ ]:
df_pol.tail()

In [ ]:
df_business.head()

## import txt files

In [ ]:
# import txt file
# path = os.getcwd() 
txt_files = glob.glob(os.path.join(path, "*/*.txt")) 
print(txt_files[0])
i = txt_files[0]
df_stops = pd.read_csv(i)
print(df_stops.shape)
print(df_stops.columns)
df_stops.tail()

## read the shp file from catchments

In [ ]:
# get the shp files
print(path)
shp_files = glob.glob(os.path.join(path, "*/*/*.shp")) 

# create empty list
dataframes_list_shp = []
 
# append datasets into the list
for i in shp_files:
    print(i)
    temp_df = gpd.read_file(i)
    dataframes_list_shp.append(temp_df)

df_catch_f = dataframes_list_shp[0]
df_catch_s = dataframes_list_shp[1]
df_catch_p = dataframes_list_shp[2]
df_sa2 = dataframes_list_shp[3]
print("catchment future: ",df_catch_f.shape,'; ',df_catch_f.columns)
print('catchment secondary: ',df_catch_s.shape,'; ',df_catch_s.columns)
print('catchment primary: ',df_catch_p.shape,'; ',df_catch_p.columns)
print('sa2: ',df_sa2.shape,'; ',df_sa2.columns)

df_catch_f.head()

In [ ]:
df_sa2.head()

In [ ]:
df_catch_s['PRIORITY'] = 'None'
df_catch_s1 = df_catch_s[['USE_ID', 'CATCH_TYPE', 'USE_DESC', 'ADD_DATE', 'KINDERGART', 'YEAR1',
       'YEAR2', 'YEAR3', 'YEAR4', 'YEAR5', 'YEAR6', 'YEAR7', 'YEAR8', 'YEAR9',
       'YEAR10', 'YEAR11', 'YEAR12', 'PRIORITY', 'geometry']]

In [ ]:
df_school = pd.concat([df_catch_f, df_catch_s1,df_catch_p])
print(df_school.shape)
df_school.tail()

In [ ]:
#transform to multipolygon
srid = 4326

def create_wkt_element(geom, srid):
    if geom.geom_type == 'Polygon':
        geom = MultiPolygon([geom])
    return WKTElement(geom.wkt, srid)

df_schoolog = df_school.copy()  # creating a copy of the original for laterog = df_school.copy()  # creating a copy of the original for later
df_school['geom'] = df_school['geometry'].apply(lambda x: create_wkt_element(geom=x,srid=srid))  # applying the function
df_school = df_school.drop(columns="geometry")  # deleting the old copy
df_school

In [ ]:
# df_sa2['geometry'].geom_type == 'Polygon'
df_sa2.dtypes

In [ ]:
df_sa2og = df_sa2.copy()  # creating a copy of the original for laterog = df_sa2.copy()  # creating a copy of the original for later
df_sa2 = df_sa2[df_sa2.geometry.type == 'Polygon']
df_sa2['geom'] = df_sa2['geometry'].apply(lambda x: create_wkt_element(geom=x,srid=srid))  # applying the function
df_sa2 = df_sa2.drop(columns="geometry")  # deleting the old copy
df_sa2['SA2_CODE21'] = df_sa2['SA2_CODE21'].astype(int)
df_sa2 

# 2. data validations

In [ ]:
## check data types, nulls, duplicates, unique values
df_school.dtypes

In [ ]:
print(df_school.shape)
print(df_school.drop_duplicates().shape)

print(df_business.shape)
print(df_business.drop_duplicates().shape)

print(df_income.shape)
print(df_income.drop_duplicates().shape)

print(df_pol.shape)
print(df_pol.drop_duplicates().shape)

print(df_pop.shape)
print(df_pop.drop_duplicates().shape)
# no duplicated rows

In [ ]:
print(df_school.isna().sum())
df_school.nunique()

In [ ]:
print(df_business.dtypes)
print(df_business.isna().sum())
df_business.nunique()

In [ ]:
print(df_pol.dtypes)
print(df_pol.isna().sum())
df_pol.nunique()

In [ ]:
print(df_pop.isna().sum())
df_pop.nunique()

In [ ]:
print(df_business.dtypes)
print(df_business.isna().sum())
df_business.nunique()

In [ ]:
# df_income.dtypes
df_income.head()

In [ ]:
# Change column A's values to floats
# df_income['earners'].sort_values().unique
df_income[df_income.earners == 'np']

In [ ]:
# keep columns only when valid data exists -- drop 7 columns due to no valid value in earners and median age/income
# Change column values to integers
df_income = df_income[df_income.earners != 'np'].astype({'earners': int, 'median_age': int,'median_income': int,'mean_income': int})


print(df_income.dtypes)

# 3. import into sql 

In [ ]:
conn.execute("""
DROP TABLE IF EXISTS polls;
CREATE TABLE polls (
    FID                             varchar(100),
state                           varchar(100),
division_id                      integer primary key,
division_name                   varchar(100),
polling_place_id                 integer,
polling_place_type_id            integer,
polling_place_name              varchar(100),
premises_name                   varchar(100),
premises_address_1              varchar(100),
premises_address_2              varchar(100),
premises_address_3              varchar(100),
premises_suburb                 varchar(100),
premises_state_abbreviation     varchar(100),
premises_post_code             integer,
latitude                       float,
longitude                      float,
the_geom                       GEOMETRY(POINT,4326)
);"""
)
df_pol.to_sql('polls', conn, if_exists='replace', index=False, dtype={'geom': Geometry('POINT', srid)})
query(conn, "select * from polls")

conn.execute("""
DROP TABLE IF EXISTS populations;
CREATE TABLE populations (
    pop_est NUMERIC, 
    continent VARCHAR(80), 
    name VARCHAR(80), 
    iso_a3 VARCHAR(80), 
    gdp_md_est NUMERIC,
    geom GEOMETRY(MULTIPOLYGON,4326) primary key
);"""
)
df_pop.to_sql('populations', conn, if_exists='replace', index=False, dtype={'geom': Geometry('POINT', srid)})
query(conn, "select * from populations")

conn.execute("""
DROP TABLE IF EXISTS incomes;
CREATE TABLE incomes (
    sa2_code21        integer primary key,
sa2_name         varchar(100),
earners          integer,
median_age       integer,
median_income    integer,
mean_income      integer
);"""
)
df_income.to_sql('incomes', conn, if_exists='replace', index=False, dtype={'geom': Geometry('POINT', srid)})
query(conn, "select * from incomes")

conn.execute("""
DROP TABLE IF EXISTS schools;
CREATE TABLE schools (
USE_ID        integer,
CATCH_TYPE    varchar(100),
USE_DESC      varchar(100),
ADD_DATE      date,
KINDERGART    varchar(100),
YEAR1         varchar(100),
YEAR2         varchar(100),
YEAR3         varchar(100),
YEAR4         varchar(100),
YEAR5         varchar(100),
YEAR6         varchar(100),
YEAR7         varchar(100),
YEAR8         varchar(100),
YEAR9         varchar(100),
YEAR10        varchar(100),
YEAR11        varchar(100),
YEAR12        varchar(100),
PRIORITY      varchar(100),
    geom GEOMETRY(MULTIPOLYGON,4326) primary key
);"""
)
df_school.to_sql('schools', conn, if_exists='replace', index=False, dtype={'geom': Geometry('MULTIPOLYGON', srid)})
query(conn, "select * from schools")


conn.execute("""
DROP TABLE IF EXISTS business;
CREATE TABLE business (
    industry_code             varchar(100),
    industry_name             varchar(100),
    sa2_code                   integer,
    sa2_name                  varchar(100),
    "0_to_50k_businesses"       integer,
    "50k_to_200k_businesses"     integer,
    "200k_to_2m_businesses"      integer,
    "2m_to_5m_businesses"        integer,
    "5m_to_10m_businesses"       integer,
    "10m_or_more_businesses"     integer,
    total_businesses           integer
);"""
)
df_business.to_sql('business', conn, if_exists='replace', index=False, dtype={'geom': Geometry('POINT', srid)})
query(conn, "select * from business")


conn.execute("""
DROP TABLE IF EXISTS stops;
CREATE TABLE stops (
stop_id int primary key, 
stop_code int, 
stop_name varchar(100), 
stop_lat float, 
stop_lon float,
location_type int, 
parent_station int, 
wheelchair_boarding int,
platform_code varchar(100)
);"""
)
df_stops.to_sql('stops', conn, if_exists='replace', index=False)
query(conn, "select * from stops")


conn.execute("""
DROP TABLE IF EXISTS sa2;
CREATE TABLE sa2 (
SA2_CODE21      integer primary key,
SA2_NAME21      varchar(100),
CHG_FLAG21      varchar(100),
CHG_LBL21       varchar(100),
SA3_CODE21      integer,
SA3_NAME21      varchar(100),
SA4_CODE21      integer,
SA4_NAME21      varchar(100),
GCC_CODE21      varchar(100),
GCC_NAME21      varchar(100),
STE_CODE21      integer,
STE_NAME21      varchar(100),
AUS_CODE21      varchar(100),
AUS_NAME21      varchar(100),
AREASQKM21     float,
LOCI_URI21      varchar(100),
geometry      geometry(Multipolygon,4326)
             );"""
)
df_sa2.to_sql('sa2', conn, if_exists='replace', index=False, dtype={'geom': Geometry('Multipolygon', srid)})
query(conn, "select * from sa2")

In [ ]:
df_business

## Task 2
Compute a score for how ”well-resourced” each individual neighbourhood is according to the formula provided on the next page, where S is the sigmoid function, z is the normalised z-score, and ’young people’ are defined as anyone aged 0-19.


### 2.1 Merge into 1 big dataset based on SA2Codes (except schools)

In [ ]:
conn.execute('''
drop table if exists combined;
    create table combined as
	with polls as (select *,ST_SetSRID(st_makepoint("longitude","latitude"),4326) as geom 
    			from polls
			   where the_geom is not null )
    ,stops as (select *,ST_SetSRID(st_makepoint("stop_lon","stop_lat"),4326) as geom 
    			from stops
			   where stop_lat is not null and stop_lon is not null)
	,join_polls as (select s."SA2_CODE21",s."SA2_NAME21",s."SA3_NAME21",s."SA4_NAME21","STE_NAME21",s."GCC_NAME21", s.geom,ST_Area(s.geom) * 1000000 as area_size,count(distinct p.*) as num_polls
				   from sa2 s
				   left join polls p on st_contains(s.geom, p.geom) -- 2505
				   
					group by 1,2,3,4,5,6,7,8
				   )
	,join_schools as (
		select s."SA2_CODE21",s."SA2_NAME21",s."SA3_NAME21",s."SA4_NAME21","STE_NAME21",s."GCC_NAME21", s.geom, s.area_size, num_polls,count(distinct sch."USE_ID") as num_schools
				   from join_polls s
				   left join schools sch on st_intersects(s.geom,sch.geom)
					  group by 1,2,3,4,5,6,7,8,9
				   )
	,join_stops as (select s."SA2_CODE21",s."SA2_NAME21",s."SA3_NAME21",s."SA4_NAME21","STE_NAME21",s."GCC_NAME21", s.geom, s.area_size, num_polls,num_schools,count(distinct stop_id) as num_stops
				   from join_schools s
				   left join stops on st_contains(s.geom, stops.geom)
					  group by 1,2,3,4,5,6,7,8,9,10
				   )
select 
s."SA2_CODE21",s."SA2_NAME21",s."SA3_NAME21",s."SA4_NAME21","STE_NAME21",s."GCC_NAME21", s.geom, s.area_size, num_polls,num_schools,num_stops
,sum(i.earners) as num_earners
,avg(i.median_age) as median_age
,avg(i.median_income) as median_income
,avg(i.mean_income) as mean_income
,sum(b.total_businesses) as total_businesses
,sum("0_to_50k_businesses" * 25 + "50k_to_200k_businesses" * 125 + "200k_to_2m_businesses" * 1100 + "2m_to_5m_businesses" * 3500 + "5m_to_10m_businesses" * 7500 + "10m_or_more_businesses" * 12000) as Total_turnover
,sum(ppl."0-4_people"+ppl."5-9_people" + ppl."10-14_people"+ppl."15-19_people") as num_0_19_people
,case when sum("total_people") =0 then null else sum(ppl."0-4_people"+ppl."5-9_people" + ppl."10-14_people"+ppl."15-19_people") / sum(ppl."total_people") end as Young_perc
,sum("total_people") total_people

from join_stops s
left join business b on b.sa2_code = s."SA2_CODE21" -- inner 11229
left join populations ppl on ppl.sa2_code = s."SA2_CODE21"-- inner 359
left join incomes i on i."sa2_code21" = s."SA2_CODE21" --inner 591
where num_polls is not null or num_schools is not null or num_stops is not null
or i.earners is not null or b.total_businesses is not null or ppl.total_people is not null
group by 1,2,3,4,5,6,7,8,9,10,11
''')
df_combined = query(conn, "select * from combined")

In [ ]:
df_combined.head()

In [ ]:
df_combined.dtypes

### check columns and understand the ranges

In [ ]:
df_combined.describe()

### check distributions

In [ ]:
df_combined['area_size'].hist(bins=100)

# add labels and title
plt.xlabel('area_size')
plt.ylabel('Frequency')
plt.title('Distribution of area_size')
# long tail

In [ ]:
df_combined['num_polls'].hist(bins=100)

# add labels and title
plt.xlabel('num_polls')
plt.ylabel('Frequency')
plt.title('Distribution of num_polls')
# long tail

In [ ]:
df_combined['num_schools'].hist(bins=100)

# add labels and title
plt.xlabel('num_schools')
plt.ylabel('Frequency')
plt.title('Distribution of num_schools')

# long tail

In [ ]:
df_combined['num_stops'].hist(bins=100)

# add labels and title
plt.xlabel('num_stops')
plt.ylabel('Frequency')
plt.title('Distribution of num_stops')

# long tail

In [ ]:
df_combined['num_earners'].hist(bins=100)

# add labels and title
plt.xlabel('num_earners')
plt.ylabel('Frequency')
plt.title('Distribution of num_earners')
# quite even

In [ ]:
df_combined['median_age'].hist(bins=100)

# add labels and title
plt.xlabel('median_age')
plt.ylabel('Frequency')
plt.title('Distribution of median_age')
# quite even

In [ ]:
df_combined['median_income'].hist(bins=100)

# add labels and title
plt.xlabel('median_income')
plt.ylabel('Frequency')
plt.title('Distribution of median_income')
# long tail on both side

In [ ]:
df_combined['total_businesses'].hist(bins=100)

# add labels and title
plt.xlabel('total_businesses')
plt.ylabel('Frequency')
plt.title('Distribution of total_businesses')
# long tail

In [ ]:
df_combined['total_turnover'].hist(bins=100)

# add labels and title
plt.xlabel('total_turnover')
plt.ylabel('Frequency')
plt.title('Distribution of total_turnover')
# long tail 

In [ ]:
df_combined['num_0_19_people'].hist(bins=100)

# add labels and title
plt.xlabel('num_0_19_people')
plt.ylabel('Frequency')
plt.title('Distribution of num_0_19_people')
# looks good?

In [ ]:
df_combined['young_perc'].hist(bins=100)

# add labels and title
plt.xlabel('young_perc')
plt.ylabel('Frequency')
plt.title('Distribution of young_perc')
# skewed?

In [ ]:
df_combined['total_people'].hist(bins=1000)

# add labels and title
plt.xlabel('total_people')
plt.ylabel('Frequency')
plt.title('Distribution of total_people')
# skewed?

In [ ]:
df_combined[df_combined.num_earners.notnull()].describe()

In [ ]:
df_combined[df_combined.young_perc.notnull()].describe()

In [ ]:
df_combined[df_combined.total_people>=100].describe()

## 2.2 Create Z-scores

In [ ]:
conn.execute(
	'''
	drop table if exists combined_z_score;
    create table combined_z_score as

    with stats as (
    select avg(num_earners/area_size) as earners_avg
    ,stddev(num_earners/area_size) as earners_stdev
    ,avg(median_age) as age_avg
    ,stddev(median_age) as age_stdev
    ,avg(median_income) as income_avg
    ,stddev(median_income) as income_stdev
    ,avg(total_businesses/area_size) as num_business_avg
    ,stddev(total_businesses/area_size) as num_business_stdev
    ,avg(total_turnover/area_size) as turnover_avg
    ,stddev(total_turnover/area_size) as turnover_stdev
    ,avg(num_0_19_people/area_size) as young_num_avg
    ,stddev(num_0_19_people/area_size) as young_num_stdev
    ,avg(Young_perc) as young_perc_avg
    ,stddev(Young_perc) as young_perc_stdev
    ,avg(num_polls/area_size) as polls_avg
    ,stddev(num_polls/area_size) as polls_stdev
    ,avg(case when num_0_19_people = 0 then 0 else num_schools/num_0_19_people*1000 end) as schools_avg
    ,stddev(case when num_0_19_people = 0 then 0 else num_schools/num_0_19_people*1000 end) as schools_stdev
    ,avg(num_stops/area_size) as stops_avg
    ,stddev(num_stops/area_size) as stops_stdev
    from combined
    where total_people >= 100
    )
    select "SA2_CODE21","SA2_NAME21","SA3_NAME21","SA4_NAME21","STE_NAME21"
    , (num_earners/area_size - earners_avg)/earners_stdev as earners_zscore
    , (median_age - age_avg)/age_stdev as age_zscore
    , (median_income - income_avg)/income_stdev as income_zscore
    , (total_businesses/area_size - num_business_avg)/num_business_stdev as business_zscore
    , (total_turnover/area_size - turnover_avg)/turnover_stdev as turnover_zscore
    , (num_0_19_people/area_size - young_num_avg)/young_num_stdev as yound_population_zscore
    , (young_perc - young_perc_avg)/young_perc_stdev as young_perc_zscore
    , (num_polls/area_size - polls_avg)/polls_stdev as polls_zscore
    , (case when num_0_19_people = 0 then 0 else num_schools/num_0_19_people*1000 end - schools_avg)/schools_stdev as schools_zscore
    , (num_stops/area_size - stops_avg)/stops_stdev as stops_zscore
    , geom
    from combined
		join stats s on 1=1
    where total_people >=100
		
    ''')
df_combined_z_score = query(conn, "select * from combined_z_score")
df_combined_z_score.head()

In [ ]:
df_combined_z_score.dtypes

In [ ]:
df_combined_z_score.describe()

## 2.3 Sigmoid Function

In [ ]:
# import math

# def sigmoid(x):
#   return 1 / (1 + math.exp(-x))
import numpy as np

def sigmoid(x):  
    return np.exp(-np.logaddexp(0, -x))

In [ ]:
df_combined_z_score['sigmoid_result'] =sigmoid(df_combined_z_score['earners_zscore'] 
                                    + df_combined_z_score['age_zscore']                 
                                    + df_combined_z_score['income_zscore']              
                                    + df_combined_z_score['business_zscore']            
                                    + df_combined_z_score['turnover_zscore']            
                                    + df_combined_z_score['yound_population_zscore']    
                                    + df_combined_z_score['young_perc_zscore']          
                                    + df_combined_z_score['polls_zscore']               
                                    + df_combined_z_score['schools_zscore']             
                                    + df_combined_z_score['stops_zscore'] )

In [ ]:
df_combined_z_score[['geom','sigmoid_result']].describe()

In [ ]:
df_combined_z_score.to_sql('task_2_result', conn, if_exists='replace', index=False, dtype={'geom': Geometry('Multipolygon', srid)})
query(conn, "select * from task_2_result")

### Push to SQL

# plot

In [ ]:
task2 = gpd.read_postgis('''select sigmoid_result,geom from task_2_result''', conn, crs=4326)
task2.plot(figsize=(20, 20),cmap='Set2',categorical=True,
                legend=True,)

In [1]:
# !pip install polars
from shapely.geometry import Point
import polars as pl

In [ ]:
(
    ggplot()
    + geom_map(task2, fill=None)
    # + geom_map(alerts_geo_df, aes(fill="alert_solved"), size=2)
    # + geom_map(stations_geo_df, colour="yellow", size=3)
    + labs(
        title="Alerts solved y/n, by location + Stations (yellow)",
        caption = "Data from  2023-01-01 to 2023-02-01",
    )
)

In [ ]:
# def calc_color(data, color=None):
#         if color   == 1: 
#             color_sq =  ['#dadaebFF','#bcbddcF0','#9e9ac8F0','#807dbaF0','#6a51a3F0','#54278fF0']; 
#             colors = 'Purples';
#         elif color == 2: 
#             color_sq = ['#c7e9b4','#7fcdbb','#41b6c4','#1d91c0','#225ea8','#253494']; 
#             colors = 'YlGnBu';
#         elif color == 3: 
#             color_sq = ['#f7f7f7','#d9d9d9','#bdbdbd','#969696','#636363','#252525']; 
#             colors = 'Greys';
#         elif color == 9: 
#             color_sq = ['#ff0000','#ff0000','#ff0000','#ff0000','#ff0000','#ff0000'];
                        
#         else:           
#             color_sq = ['#ffffd4','#fee391','#fec44f','#fe9929','#d95f0e','#993404']; 
#             colors = 'YlOrBr';
#         new_data, bins = pd.qcut(data, 6, retbins=True, 
#         labels=list(range(6)))
#         color_ton = []
#         for val in new_data:
#             color_ton.append(color_sq[val]) 
#         if color != 9:
#             colors = sns.color_palette(colors, n_colors=6)
#             sns.palplot(colors, 0.6);
#             for i in range(6):
#                 print ("\n"+str(i+1)+': '+str(int(bins[i]))+
#                        " => "+str(int(bins[i+1])-1))
#             print("\n\n   1   2   3   4   5   6")    
#         return color_ton, bins

In [ ]:
# def plot_cities_data(sf, title, cities, data=None,color=None, print_id=False):
 
#     color_ton, bins = calc_color(data, color)
#     df = read_shapefile(sf)
#     city_id = []
#     for i in cities:
#         city_id.append(df[df.DIST_NAME == 
#                             i.upper()].index.get_values()[0])
#     plot_map_fill_multiples_ids_tone(sf, title, city_id, 
#                                      print_id, 
#                                      color_ton, 
#                                      bins, 
#                                      x_lim = None, 
#                                      y_lim = None, 
#                                      figsize = (11,9));
# def plot_map_fill_multiples_ids_tone(sf, title, city,  
#                                      print_id, color_ton, 
#                                      bins, 
#                                      x_lim = None, 
#                                      y_lim = None, 
#                                      figsize = (11,9)):
   
        
#     plt.figure(figsize = figsize)
#     fig, ax = plt.subplots(figsize = figsize)
#     fig.suptitle(title, fontsize=16)
#     for shape in sf.shapeRecords():
#         x = [i[0] for i in shape.shape.points[:]]
#         y = [i[1] for i in shape.shape.points[:]]
#         ax.plot(x, y, 'k')
            
#     for id in city:
#         shape_ex = sf.shape(id)
#         x_lon = np.zeros((len(shape_ex.points),1))
#         y_lat = np.zeros((len(shape_ex.points),1))
#         for ip in range(len(shape_ex.points)):
#             x_lon[ip] = shape_ex.points[ip][0]
#             y_lat[ip] = shape_ex.points[ip][1]
#         ax.fill(x_lon,y_lat, color_ton[city.index(id)])
#         if print_id != False:
#             x0 = np.mean(x_lon)
#             y0 = np.mean(y_lat)
#             plt.text(x0, y0, id, fontsize=10)
#     if (x_lim != None) & (y_lim != None):     
#         plt.xlim(x_lim)
#         plt.ylim(y_lim)


# task 2

In [ ]:
df = query(conn, "select * from combined where total_people is null or total_people = 'NaN'")
df.columns

In [ ]:
df[['SA2_CODE21', 'SA2_NAME21','0-4_people',
       '5-9_people', '10-14_people', '15-19_people', '20-24_people',
       '25-29_people', '30-34_people', '35-39_people', '40-44_people',
       '45-49_people', '50-54_people', '55-59_people', '60-64_people',
       '65-69_people', '70-74_people', '75-79_people', '80-84_people',
       '85-and-over_people', 'total_people']]

In [ ]:
df.sum('0-4_people')

#  Task 3

# POI

In [ ]:
path_to_file = '/Users/apple/Desktop/Data2901/Assignment/DATA2901--2024S1/Data/Points_Of_Interest_EPSG4326.json'

In [ ]:
import geojson
with open(path_to_file) as f:
    gj = geojson.load(f)
# features = gj['features'][0]

In [ ]:
gj.dtypes